# **SETUP**

In [102]:
from collections import defaultdict

import cv2
import numpy as np


import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib import rc as rc
import IPython.display
import matplotlib as mpl

# Set the animation embed limit to 50 MB
mpl.rcParams['animation.embed_limit'] = 1000
rc('animation', html='jshtml')

# from google.colab import drive
# drive.mount('/content/drive')

# **VIDEO PLAYER**

In [ ]:
def draw_text(img, text,
          pos=(0, 0),
          font=cv2.FONT_HERSHEY_PLAIN,
          font_scale=3.0,
          font_thickness=2,
          text_color=(0, 255, 0),
          text_color_bg=(0, 0, 0)
          ):

    x, y = pos
    text_size, _ = cv2.getTextSize(text, font, font_scale, font_thickness)
    text_w, text_h = text_size
    cv2.rectangle(img, pos, (x + text_w, y + text_h), text_color_bg, -1)
    cv2.putText(img, text, (x, round(y + text_h + font_scale - 1)), font, font_scale, text_color, font_thickness)

    return text_size

In [ ]:
class VideoPlayer:
    def __init__(self, video_path):
        self.video_path = video_path
        self.cap = cv2.VideoCapture(video_path)
        if not self.cap.isOpened():
            raise ValueError(f"Could not open video file: {video_path}")

        self.frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.figsize = (5, round(5 / self.width * self.height))

        print(self.figsize)

        # Close immediately if you only need the metadata
        self.cap.release()

    def get_frames(self, start_frame, end_frame):
        """
        Returns all frames in RGB from start_frame (inclusive) to end_frame (exclusive).
        """
        if start_frame >= self.frame_count:
            return np.array([])

        cap = cv2.VideoCapture(self.video_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        frames = []
        for frame_idx in range(start_frame, end_frame):
            if frame_idx >= self.frame_count:
                break
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_rgb)
        cap.release()
        return np.array(frames)

    def plot(
        self,
        start_frame=0,
        end_frame=100,
        bounding_boxes_gt=None,
        bounding_boxes_pred=None,
        gt_color=(0, 255, 0),
        pred_color=(255, 0, 0),
        bbox_thickness=5,
        save_file=None,
        taking_only_GTobject_into_account=False,
        iou_threshold=0.2,
    ):
        """
        Displays an animation of the video frames within [start_frame, end_frame).
        Optionally draws bounding boxes for ground-truth and predicted data,
        with track IDs rendered in a small filled square of the same color.
        
        If taking_only_GTobject_into_account is True, predicted bounding boxes
        will be filtered per frame based on an IoU threshold with the GT boxes.
        """
        frames = self.get_frames(start_frame-1, end_frame-1)
        if len(frames) == 0:
            print("No frames to display for the specified frame range.")
            return

        processed_frames = []
        label_box_size = 30  # Size (width=height) of the track ID "label box"

        for i, frame in enumerate(frames):
            # Convert to uint8 for drawing bounding boxes in OpenCV
            out_frame_uint8 = frame.astype(np.uint8)

            # Draw ground-truth bounding boxes
            if bounding_boxes_gt is not None:
                for gt_box in bounding_boxes_gt[i + start_frame]:
                    x, y, w, h = gt_box["bbox"]
                    # Draw the bounding box
                    cv2.rectangle(
                        out_frame_uint8,
                        (int(x), int(y)), (int(x + w), int(y + h)),
                        color=gt_color,
                        thickness=bbox_thickness
                    )
                    # Draw the track_id box (if present)
                    if "track_id" in gt_box:
                        tid = str(gt_box["track_id"])
                        # Draw a small filled square near the top-left of the box
                        lx, ly = int(x + 0.25 * w), int(y + h + 5)
                        draw_text(
                            out_frame_uint8,
                            tid,
                            pos=(lx, ly),
                            font=cv2.FONT_HERSHEY_SIMPLEX,
                            font_scale=1.25,
                            text_color=(0, 0, 0),
                            font_thickness=3,
                            text_color_bg=gt_color
                        )

            # Draw predicted bounding boxes
            if bounding_boxes_pred is not None:
                for pred_box in bounding_boxes_pred[i + start_frame]:
                    # If filtering flag is active and GT boxes exist, check IoU overlap.
                    if taking_only_GTobject_into_account and (bounding_boxes_gt is not None):
                        current_gt_boxes = bounding_boxes_gt[i + start_frame]
                        # Skip drawing this pred_box if no sufficient overlap is found.
                        if not any(iou(pred_box["bbox"], gt_box["bbox"]) >= iou_threshold for gt_box in current_gt_boxes):
                            continue

                    x, y, w, h = pred_box["bbox"]
                    cv2.rectangle(
                        out_frame_uint8,
                        (int(x), int(y)), (int(x + w), int(y + h)),
                        color=pred_color,
                        thickness=bbox_thickness
                    )
                    # Draw the track_id box (if present)
                    if "track_id" in pred_box:
                        tid = str(pred_box["track_id"])
                        # Draw a small filled square near the top-right of the box
                        lx, ly = int(x + 0.75 * w), int(y + h + 5)
                        draw_text(
                            out_frame_uint8,
                            tid,
                            pos=(lx, ly),
                            font=cv2.FONT_HERSHEY_SIMPLEX,
                            font_scale=1.25,
                            text_color=(0, 0, 0),
                            font_thickness=3,
                            text_color_bg=pred_color
                        )

            processed_frames.append(out_frame_uint8)

        processed_frames = np.array(processed_frames)

        # Create the Matplotlib animation
        fig, ax = plt.subplots(figsize=self.figsize)
        img_display = ax.imshow(processed_frames[0])
        ax.set_title(f"Frame: {start_frame + 1}/{self.frame_count}")
        ax.axis("off")
        plt.tight_layout()

        def update(frame_idx):
            img_display.set_data(processed_frames[frame_idx])
            ax.set_title(f"Frame: {start_frame + frame_idx + 1}/{self.frame_count}")
            return (img_display,)

        anim = FuncAnimation(
            fig,
            update,
            frames=len(processed_frames),
            interval=1000 / self.fps if self.fps else 40,
            blit=True
        )

        if save_file is not None:
            from matplotlib.animation import FFMpegWriter, PillowWriter
            ext = save_file.lower().rsplit('.', 1)[-1]
            fps_for_save = int(self.fps) if self.fps else 30

            if ext == 'gif':
                writer = PillowWriter(fps=fps_for_save)
            else:
                writer = FFMpegWriter(fps=fps_for_save, metadata={'title': 'Video Output'})

            anim.save(save_file, writer=writer)
            print(f"Animation saved to {save_file}")
        else:
            IPython.display.display(anim)
            plt.close(fig)

# **I/O**

## XML

In [ ]:
def parse_cvat_annotations(xml_path):
    """
    Reads a CVAT annotation XML file and returns ground truth data
    in a dict of the form:
        {
            frame_idx: [
                {
                    "bbox": [x, y, w, h],
                    "category_id": <int>,
                    "track_id": <str>
                },
                ...
            ],
            ...
        }

    Assumptions/notes:
      - Only 'outside="0"' (visible) objects will be returned.
      - Only non-parked objects are returned (based on <attribute name="parked">).
      - The default category_id is given by label_to_id below; unrecognized labels
        will get new IDs assigned automatically.
      - Frame indices are taken from the 'box' element's 'frame' attribute in CVAT.
    """

    # Manually seed known labels if you like,
    # or leave empty and assign new IDs on the fly:
    label_to_id = {
        "car": 1,
        "bike": 1,
    }

    frames_dict = defaultdict(list)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Iterate over each "track" element in <annotations>
    for track in root.findall("track"):
        label = track.attrib["label"]  # e.g. "car" or "bike"
        if label not in label_to_id:
            # Assign a new ID if the label is not recognized
            label_to_id[label] = len(label_to_id) + 1

        track_id = track.attrib["id"]  # track identifier as string

        for box in track.findall("box"):
            frame_str = box.attrib["frame"]
            xtl = float(box.attrib["xtl"])
            ytl = float(box.attrib["ytl"])
            xbr = float(box.attrib["xbr"])
            ybr = float(box.attrib["ybr"])
            outside = box.attrib["outside"]  # "0" or "1"

            # Check whether this box is marked 'parked'
            parked = False
            for attr_node in box.findall("attribute"):
                if attr_node.attrib.get("name") == "parked":
                    if attr_node.text.strip().lower() == "true":
                        parked = True
                        break

            # Skip 'outside' (invisible) objects
            if outside == "1" or label=='bike':
                continue

            # Convert to [x, y, w, h]
            x = xtl
            y = ytl
            w = xbr - xtl
            h = ybr - ytl

            frame_idx = int(frame_str)
            cat_id = label_to_id[label]

            annotation = {
                "bbox": [x, y, w, h],
                "category_id": cat_id,
                "track_id": track_id,
                "conf": 1,
            }

            frames_dict[frame_idx].append(annotation)
    return frames_dict

## TXT

In [ ]:
def parse_tracking_file(filepath):
    """
    Reads the detection text file and returns data grouped by frame.
    Each element in the returned list corresponds to a single frame,
    which itself is a list of dictionaries.
    Each dictionary has keys: 'bbox' -> [left, top, width, height], 'conf' -> conf_value

    :param filepath: Path to the input text file.
    """

    # Using a dictionary to accumulate detections by frame number:
    frames_dict = defaultdict(list)

    with open(filepath, 'r') as f:
        for line in f:
            # Strip and skip any empty lines
            line = line.strip()
            if not line:
                continue

            # Split line into fields
            fields = line.split(',')
            # fields are expected as: frame, track_id, left, top, width, height, conf, -1, -1, -1

            frame = int(fields[0].strip())
            track_id = int(fields[1].strip())
            left  = float(fields[2].strip())
            top   = float(fields[3].strip())
            width = float(fields[4].strip())
            height= float(fields[5].strip())
            conf  = float(fields[6].strip())

            # Construct detection dictionary
            detection = {
                'bbox': [left, top, width, height],
                'conf': conf,
                'track_id': track_id
            }

            # Append the detection to the corresponding frame
            frames_dict[frame].append(detection)

    return frames_dict


def save_tracking_data(filepath, tracking_data):
    """
    Saves tracking data to a file in the format:
      frame,id,left,top,width,height,conf,-1,-1,-1

    :param filepath: Path to the output text file.
    :param tracking_data: A list of frames (list),
                          where each frame is a list of dictionaries.
                          Each dictionary has keys:
                            {
                                'id': <integer ID>,
                                'bbox': [left, top, width, height],
                                'conf': <float confidence>
                            }
    """
    with open(filepath, 'w') as f:
        # 'frame_idx' will start from 1, but adjust if your frames are 0-based
        for frame_idx, detections in enumerate(tracking_data, start=1):
            for det in detections:
                box_id = det['track_id']
                left, top, width, height = det['bbox']
                conf = det['conf']
                # Write one line per detection
                line = f"{frame_idx},{box_id},{left:.2f},{top:.2f},{width:.2f},{height:.2f},{conf:.2f},-1,-1,-1"
                f.write(line + "\n")

## Utils

In [ ]:
def iou(box_a, box_b):
    """
    Computes the Intersection-over-Union (IoU) of two boxes.
    Each box is in the format [x, y, w, h].
    """
    # Convert [x, y, w, h] to (xmin, ymin, xmax, ymax)
    ax1, ay1 = box_a[0], box_a[1]
    ax2, ay2 = ax1 + box_a[2], ay1 + box_a[3]

    bx1, by1 = box_b[0], box_b[1]
    bx2, by2 = bx1 + box_b[2], by1 + box_b[3]

    # Intersection rectangle
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    intersection_area = inter_w * inter_h

    # Areas of each box
    area_a = box_a[2] * box_a[3]  # w*h
    area_b = box_b[2] * box_b[3]

    union_area = area_a + area_b - intersection_area
    if union_area == 0:
        return 0.0
    return intersection_area / union_area


def union_box(box_a, box_b):
    """
    Returns the bounding-box union of the two boxes
    (the minimal rectangle that encloses both).
    Each box is in the format [x, y, w, h].
    """
    ax1, ay1 = box_a[0], box_a[1]
    ax2, ay2 = ax1 + box_a[2], ay1 + box_a[3]

    bx1, by1 = box_b[0], box_b[1]
    bx2, by2 = bx1 + box_b[2], by1 + box_b[3]

    union_x1 = min(ax1, bx1)
    union_y1 = min(ay1, by1)
    union_x2 = max(ax2, bx2)
    union_y2 = max(ay2, by2)

    return [union_x1, union_y1, union_x2 - union_x1, union_y2 - union_y1]


def union_of_boxes(list_of_boxes):
    """
    Given a list of [x, y, w, h] boxes, returns the bounding box that encloses them all.
    """
    if not list_of_boxes:
        return None

    # Initialize x1, y1 with a large value, x2, y2 with a small value
    x1 = float('inf')
    y1 = float('inf')
    x2 = float('-inf')
    y2 = float('-inf')

    for (x, y, w, h) in list_of_boxes:
        # Convert [x, y, w, h] into corners
        bx1, by1 = x, y
        bx2, by2 = x + w, y + h

        # Update union coordinates
        x1 = min(x1, bx1)
        y1 = min(y1, by1)
        x2 = max(x2, bx2)
        y2 = max(y2, by2)

    # Convert corners back to [x, y, w, h]
    return [x1, y1, x2 - x1, y2 - y1]


def merge_overlapping_boxes(bboxes, iou_threshold=0.5):
    """
    Given a list of bounding boxes [ [x, y, w, h], ... ],
    merges any two boxes whose IoU > iou_threshold into their union.
    Repeats until no further merges are found.
    Returns the merged list of boxes.
    """
    merged = True
    boxes = bboxes[:]

    # Keep merging until no more merges happen
    while merged:
        merged = False
        new_boxes = []
        while boxes:
            current_box = boxes.pop()
            # Try to merge current_box with one of the boxes already in new_boxes
            for i, nb in enumerate(new_boxes):
                if iou(current_box['bbox'], nb['bbox']) > iou_threshold:
                    # Merge them
                    merged_box = union_box(current_box['bbox'], nb['bbox'])
                    merged_conf = max(current_box['conf'], nb['conf'])
                    # Replace the box in new_boxes with the merged box
                    new_boxes[i] = {'bbox': merged_box, 'conf': merged_conf}
                    merged = True
                    break
            else:
                # If we never broke, it means no merge happened; keep current_box
                new_boxes.append(current_box)
        boxes = new_boxes

    return boxes

### SORT Code

In [ ]:
import os
import numpy as np
from skimage import io

import glob
import time
import argparse
from filterpy.kalman import KalmanFilter

np.random.seed(0)


def linear_assignment(cost_matrix):
    from scipy.optimize import linear_sum_assignment
    x, y = linear_sum_assignment(cost_matrix)
    return np.array(list(zip(x, y)))


def iou_batch(bb_test, bb_gt):
    """
    From SORT: Computes IOU between two bboxes in the form [x1,y1,x2,y2]
    """
    bb_gt = np.expand_dims(bb_gt, 0)
    bb_test = np.expand_dims(bb_test, 1)

    xx1 = np.maximum(bb_test[..., 0], bb_gt[..., 0])
    yy1 = np.maximum(bb_test[..., 1], bb_gt[..., 1])
    xx2 = np.minimum(bb_test[..., 2], bb_gt[..., 2])
    yy2 = np.minimum(bb_test[..., 3], bb_gt[..., 3])
    w = np.maximum(0., xx2 - xx1)
    h = np.maximum(0., yy2 - yy1)
    wh = w * h
    o = wh / ((bb_test[..., 2] - bb_test[..., 0]) * (bb_test[..., 3] - bb_test[..., 1])
    + (bb_gt[..., 2] - bb_gt[..., 0]) * (bb_gt[..., 3] - bb_gt[..., 1]) - wh)
    return(o)


def convert_bbox_to_z(bbox):
    """
    Takes a bounding box in the form [x1,y1,x2,y2] and returns z in the form
    [x,y,s,r] where x,y is the centre of the box and s is the scale/area and r is
    the aspect ratio
    """
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    x = bbox[0] + w/2.
    y = bbox[1] + h/2.
    s = w * h    #scale is just area
    r = w / float(h)
    return np.array([x, y, s, r]).reshape((4, 1))


def convert_x_to_bbox(x,score=None):
    """
    Takes a bounding box in the centre form [x,y,s,r] and returns it in the form
    [x1,y1,x2,y2] where x1,y1 is the top left and x2,y2 is the bottom right
    """
    w = np.sqrt(x[2] * x[3])
    h = x[2] / w
    if(score==None):
        return np.array([x[0]-w/2.,x[1]-h/2.,x[0]+w/2.,x[1]+h/2.]).reshape((1,4))
    else:
        return np.array([x[0]-w/2.,x[1]-h/2.,x[0]+w/2.,x[1]+h/2.,score]).reshape((1,5))


class KalmanBoxTracker(object):
    """
    This class represents the internal state of individual tracked objects observed as bbox.
    """
    count = 0
    def __init__(self,bbox):
        """
        Initialises a tracker using initial bounding box.
        """
        #define constant velocity model
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([[1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],[0,0,0,1,0,0,0],  [0,0,0,0,1,0,0],[0,0,0,0,0,1,0],[0,0,0,0,0,0,1]])
        self.kf.H = np.array([[1,0,0,0,0,0,0],[0,1,0,0,0,0,0],[0,0,1,0,0,0,0],[0,0,0,1,0,0,0]])

        self.kf.R[2:,2:] *= 10.
        self.kf.P[4:,4:] *= 1000. #give high uncertainty to the unobservable initial velocities
        self.kf.P *= 10.
        self.kf.Q[-1,-1] *= 0.01
        self.kf.Q[4:,4:] *= 0.01

        self.kf.x[:4] = convert_bbox_to_z(bbox)
        self.time_since_update = 0
        self.id = KalmanBoxTracker.count
        KalmanBoxTracker.count += 1
        self.history = []
        self.hits = 0
        self.hit_streak = 0
        self.age = 0

    def update(self,bbox):
        """
        Updates the state vector with observed bbox.
        """
        self.time_since_update = 0
        self.history = []
        self.hits += 1
        self.hit_streak += 1
        self.kf.update(convert_bbox_to_z(bbox))

    def predict(self):
        """
        Advances the state vector and returns the predicted bounding box estimate.
        """
        if((self.kf.x[6]+self.kf.x[2])<=0):
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.age += 1
        if(self.time_since_update>0):
            self.hit_streak = 0
        self.time_since_update += 1
        self.history.append(convert_x_to_bbox(self.kf.x))
        return self.history[-1]

    def get_state(self):
        """
        Returns the current bounding box estimate.
        """
        return convert_x_to_bbox(self.kf.x)


def associate_detections_to_trackers(detections, trackers, iou_threshold = 0.3):
    """
    Assigns detections to tracked object (both represented as bounding boxes)

    Returns 3 lists of matches, unmatched_detections and unmatched_trackers
    """
    if(len(trackers)==0):
        return np.empty((0,2),dtype=int), np.arange(len(detections)), np.empty((0,5),dtype=int)

    iou_matrix = iou_batch(detections, trackers)

    if min(iou_matrix.shape) > 0:
        a = (iou_matrix > iou_threshold).astype(np.int32)
        if a.sum(1).max() == 1 and a.sum(0).max() == 1:
            matched_indices = np.stack(np.where(a), axis=1)
        else:
            matched_indices = linear_assignment(-iou_matrix)
    else:
        matched_indices = np.empty(shape=(0,2))

    unmatched_detections = []
    for d, det in enumerate(detections):
        if(d not in matched_indices[:,0]):
            unmatched_detections.append(d)
    unmatched_trackers = []
    for t, trk in enumerate(trackers):
        if(t not in matched_indices[:,1]):
            unmatched_trackers.append(t)

    #filter out matched with low IOU
    matches = []
    for m in matched_indices:
        if(iou_matrix[m[0], m[1]]<iou_threshold):
            unmatched_detections.append(m[0])
            unmatched_trackers.append(m[1])
        else:
            matches.append(m.reshape(1,2))
    if(len(matches)==0):
        matches = np.empty((0,2),dtype=int)
    else:
        matches = np.concatenate(matches,axis=0)

    return matches, np.array(unmatched_detections), np.array(unmatched_trackers)


class Sort(object):
  def __init__(self, max_age=1, min_hits=3, iou_threshold=0.3):
    """
    Sets key parameters for SORT
    """
    self.max_age = max_age
    self.min_hits = min_hits
    self.iou_threshold = iou_threshold
    self.trackers = []
    self.frame_count = 0

  def update(self, dets=np.empty((0, 5))):
    """
    Params:
      dets - a numpy array of detections in the format [[x1,y1,x2,y2,score],[x1,y1,x2,y2,score],...]
    Requires: this method must be called once for each frame even with empty detections (use np.empty((0, 5)) for frames without detections).
    Returns the a similar array, where the last column is the object ID.

    NOTE: The number of objects returned may differ from the number of detections provided.
    """
    self.frame_count += 1
    # get predicted locations from existing trackers.
    trks = np.zeros((len(self.trackers), 5))
    to_del = []
    ret = []
    for t, trk in enumerate(trks):
      pos = self.trackers[t].predict()[0]
      trk[:] = [pos[0], pos[1], pos[2], pos[3], 0]
      if np.any(np.isnan(pos)):
        to_del.append(t)
    trks = np.ma.compress_rows(np.ma.masked_invalid(trks))
    for t in reversed(to_del):
      self.trackers.pop(t)
    matched, unmatched_dets, unmatched_trks = associate_detections_to_trackers(dets,trks, self.iou_threshold)

    # update matched trackers with assigned detections
    for m in matched:
      self.trackers[m[1]].update(dets[m[0], :])

    # create and initialise new trackers for unmatched detections
    for i in unmatched_dets:
        trk = KalmanBoxTracker(dets[i,:])
        self.trackers.append(trk)
    i = len(self.trackers)
    for trk in reversed(self.trackers):
        d = trk.get_state()[0]
        if (trk.time_since_update < 1) and (trk.hit_streak >= self.min_hits or self.frame_count <= self.min_hits):
          ret.append(np.concatenate((d,[trk.id+1])).reshape(1,-1)) # +1 as MOT benchmark requires positive
        i -= 1
        # remove dead tracklet
        if(trk.time_since_update > self.max_age):
          self.trackers.pop(i)
    if(len(ret)>0):
      return np.concatenate(ret)
    return np.empty((0,5))

### Tracking with SORT

In [ ]:
def track_by_sort(detections_per_frame, iou_threshold=0.2, min_hits=3, max_age=1):
    """
    Uses the SORT tracker to assign track IDs to detections from frame to frame.

    :param detections_per_frame: Dictionary {frame_idx: [ { "bbox": [x1,y1,x2,y2], "conf": ... }, ... ]}
    :param iou_threshold: IoU threshold for data association in SORT.
    :param min_hits: Minimum number of hits before a track is output.
    :param max_age: Maximum frames to keep alive a track without seeing it again.
    :return: Dictionary {frame_idx: [ { "bbox": [...], "track_id": <int> }, ... ] }
             Each detection is assigned a "track_id" from SORT.
    """
    # 1) Create the SORT object with desired parameters
    tracker = Sort(max_age=max_age, min_hits=min_hits, iou_threshold=iou_threshold)

    # 2) Sort the frame indices so we process in order
    frame_indices = sorted(detections_per_frame.keys())

    # 3) Prepare the result dict
    tracking_by_frame = defaultdict(list)

    # 4) Main loop over frames
    for frame_idx in frame_indices:
        # Convert list of detection dicts -> Nx5 array: [x1, y1, x2, y2, score]
        current_detections = detections_per_frame[frame_idx]
        dets_array = []
        for det in current_detections:
            box = det["bbox"]  # Must be [x1,y1,x2,y2]
            score = det.get("conf", 1.0)  # Default to 1.0 if not provided
            dets_array.append([box[0], box[1], box[2]+box[0], box[3]+box[1], score])

        # If no detections, an empty array
        dets_array = np.array(dets_array) if len(dets_array) > 0 else np.empty((0, 5))

        # 5) Pass the detections to the SORT tracker
        tracked = tracker.update(dets_array)
        # 'tracked' is an array of shape (N, 5) -> [x1, y1, x2, y2, track_id]

        # 6) Build a list of detection dicts from the tracked results
        frame_output = []
        for t in tracked:
            x1, y1, x2, y2, track_id = t
            out_det = {
                "bbox": [float(x1), float(y1), float(x2) - float(x1), float(y2)-float(y1)],
                "track_id": int(track_id)
            }
            frame_output.append(out_det)

        tracking_by_frame[frame_idx] = frame_output

    return dict(tracking_by_frame)

# **Evaluation**

In [ ]:
from trackeval.metrics.hota import HOTA
from trackeval.metrics.identity import Identity

In [ ]:
def calculate_metrics(tracker_data, gt_data):
    # build mapping dict for tracker
    unique_tracker_ids_tr = set()
    for frame, dets in tracker_data.items():
        for det in dets:
            if 'track_id' not in det.keys():
                print(frame)
            unique_tracker_ids_tr.add(det['track_id'])
    unique_tracker_ids_tr = sorted(list(unique_tracker_ids_tr))
    tracker_id_mapping_tr = {old_id: new_id for new_id, old_id in enumerate(unique_tracker_ids_tr)}
    
    # build mapping dict for gt
    unique_tracker_ids_gt = set()
    for frame, dets in gt_data.items():
        for det in dets:
            unique_tracker_ids_gt.add(det['track_id'])
    unique_tracker_ids_gt = sorted(list(unique_tracker_ids_gt))
    tracker_id_mapping_gt = {old_id: new_id for new_id, old_id in enumerate(unique_tracker_ids_gt)}
    
    all_frames = sorted(set(gt_data.keys()).union(tracker_data.keys()))
    
    gt_ids_list = []
    tracker_ids_list = []
    similarity_scores_list = []
    total_tracker_dets = 0
    total_gt_dets = 0
    
    for frame in all_frames:
        if frame in gt_data:
            gt_dets = gt_data[frame]
            # remap track IDs for gt
            gt_ids = np.array([tracker_id_mapping_gt[det['track_id']] for det in gt_dets])
            if frame in tracker_data:
                tr_dets = tracker_data[frame]
                # remap track IDs for tr
                tr_ids = np.array([tracker_id_mapping_tr[det['track_id']] for det in tr_dets])
            else:
                tr_dets = [{'bbox': [0, 0, 0, 0],
                  'category_id': 0,
                  'track_id': 0,
                  'conf': 0}]
                tr_ids =  np.array([det['track_id'] for det in tr_dets])
            
            total_gt_dets += len(gt_dets)
            total_tracker_dets += len(tr_dets)
            
            if len(gt_dets) > 0 and len(tr_dets) > 0:
                sim_matrix = np.zeros((len(gt_dets), len(tr_dets)), dtype=float)
                for i, gt in enumerate(gt_dets):
                    for j, tr in enumerate(tr_dets):
                        sim_matrix[i, j] = iou(gt['bbox'], tr['bbox'])
            else:
                sim_matrix = np.zeros((len(gt_dets), len(tr_dets)), dtype=float)
            
            gt_ids_list.append(gt_ids)
            # gt_ids_list.append(np.array([int(a['track_id']) for a in gt_dets]))
            tracker_ids_list.append(tr_ids)
            # tracker_ids_list.append(np.array([int(a['track_id']) for a in tr_dets]))
            similarity_scores_list.append(sim_matrix)
    
    
    num_gt_ids = len(unique_tracker_ids_gt)
    num_tracker_ids = len(unique_tracker_ids_tr)
    
    # data dictionary for HOTA
    data = {
        'num_tracker_dets': total_tracker_dets,
        'num_gt_dets': total_gt_dets,
        'num_gt_ids': num_gt_ids,
        'num_tracker_ids': num_tracker_ids,
        'gt_ids': gt_ids_list,
        'tracker_ids': tracker_ids_list,
        'similarity_scores': similarity_scores_list
    }
    hota_metric = HOTA()
    identity_metric = Identity()
    result_hota = hota_metric.eval_sequence(data)
    result_identity = identity_metric.eval_sequence(data)
    return result_hota, result_identity

In [ ]:
def calculate_metrics_taking_only_GTobject_into_account(tracker_data, gt_data, iou_threshold=0.2):
    """
    Evaluates metrics using only GT objects. Instead of relying on matching IDs (since
    GT and tracker IDs differ), we filter tracker detections per frame. We retain only
    tracker detections that have an IoU >= iou_threshold with at least one GT box.
    In frames where no valid tracker detections are found, a dummy detection is inserted
    to avoid failure in the HOTA function.
    """
    # Build mapping dict for GT track IDs (from GT detections only)
    unique_gt_ids = set()
    for frame, dets in gt_data.items():
        for det in dets:
            unique_gt_ids.add(det['track_id'])
    unique_gt_ids = sorted(list(unique_gt_ids))
    gt_id_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_gt_ids)}

    # Filter tracker detections per frame using IoU threshold, only for frames present in GT
    filtered_tracker_data = {}
    for frame in gt_data.keys():
        valid_tr_dets = []
        if frame in tracker_data:
            for tr in tracker_data[frame]:
                # Compute IoU with each GT detection in this frame
                ious = [iou(gt['bbox'], tr['bbox']) for gt in gt_data[frame]]
                if ious and max(ious) >= iou_threshold:
                    valid_tr_dets.append(tr)
        # If no valid tracker detection exists, insert the dummy detection
        if not valid_tr_dets:
            valid_tr_dets = [{
                'bbox': [0, 0, 0, 0],
                'category_id': 0,
                'track_id': 0,
                'conf': 0
            }]
        filtered_tracker_data[frame] = valid_tr_dets

    # Build mapping for tracker track IDs from the filtered data
    unique_tracker_ids = set()
    for frame in gt_data.keys():
        for det in filtered_tracker_data[frame]:
            unique_tracker_ids.add(det['track_id'])
    unique_tracker_ids = sorted(list(unique_tracker_ids))
    tracker_id_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_tracker_ids)}

    # Evaluate on frames that exist in GT
    all_frames = sorted(gt_data.keys())
    gt_ids_list = []
    tracker_ids_list = []
    similarity_scores_list = []
    total_tracker_dets = 0
    total_gt_dets = 0

    for frame in all_frames:
        gt_dets = gt_data[frame]
        # Remap GT track IDs
        gt_ids = np.array([gt_id_mapping[det['track_id']] for det in gt_dets])
        tr_dets = filtered_tracker_data.get(frame, [{
            'bbox': [0, 0, 0, 0],
            'category_id': 0,
            'track_id': 0,
            'conf': 0
        }])
        tr_ids = np.array([tracker_id_mapping[det['track_id']] for det in tr_dets])
        # print(gt_ids, tr_ids)
        total_gt_dets += len(gt_dets)
        total_tracker_dets += len(tr_dets)

        # Build similarity matrix using IoU for the current frame
        if len(gt_dets) > 0 and len(tr_dets) > 0:
            sim_matrix = np.zeros((len(gt_dets), len(tr_dets)), dtype=float)
            for i, gt in enumerate(gt_dets):
                for j, tr in enumerate(tr_dets):
                    sim_matrix[i, j] = iou(gt['bbox'], tr['bbox'])
        else:
            sim_matrix = np.zeros((len(gt_dets), len(tr_dets)), dtype=float)

        gt_ids_list.append(gt_ids)
        tracker_ids_list.append(tr_ids)
        similarity_scores_list.append(sim_matrix)

    num_gt_ids = len(unique_gt_ids)
    num_tracker_ids = len(unique_tracker_ids)

    # Create the data dictionary expected by the HOTA evaluation metric
    data = {
        'num_tracker_dets': total_tracker_dets,
        'num_gt_dets': total_gt_dets,
        'num_gt_ids': num_gt_ids,
        'num_tracker_ids': num_tracker_ids,
        'gt_ids': gt_ids_list,
        'tracker_ids': tracker_ids_list,
        'similarity_scores': similarity_scores_list
    }

    hota_metric = HOTA()
    identity_metric = Identity()  # Assuming your Identity metric is defined similarly
    result_hota = hota_metric.eval_sequence(data)
    result_identity = identity_metric.eval_sequence(data)
    return result_hota, result_identity

## Last week

Note: 

* Pred: Red
* Annot: Green

In [104]:
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/S03/c010/mtsc/mtsc_tc_mask_rcnn.txt")#

annotations = parse_cvat_annotations("../ai_challenge_s03_c010-full_annotation.xml")
annotations = {k+1:v for k,v in annotations.items()}

calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.65354199, 0.65268186, 0.65243532, 0.65188141, 0.65122249,
         0.65033651, 0.64792983, 0.55362149, 0.53350805, 0.52468351,
         0.51757898, 0.5061476 , 0.49669843, 0.48872066, 0.48027648,
         0.46377606, 0.42090248, 0.287937  , 0.02271761]),
  'DetA': array([0.57640824, 0.57245716, 0.57138303, 0.5691678 , 0.56738587,
         0.56425949, 0.55804398, 0.45496711, 0.39474535, 0.3722262 ,
         0.35386668, 0.32009641, 0.29604868, 0.2798012 , 0.26729837,
         0.24831778, 0.21517115, 0.11619331, 0.0053317 ]),
  'AssA': array([0.74099761, 0.74414933, 0.74498511, 0.74661527, 0.74744675,
         0.74954448, 0.75229386, 0.67366793, 0.72104926, 0.73958467,
         0.75703089, 0.80033822, 0.83334043, 0.85363423, 0.86295137,
         0.86618136, 0.82333945, 0.7135326 , 0.09679645]),
  'DetRe': array([0.58555715, 0.58300459, 0.58230844, 0.58086973, 0.57970947,
         0.57766742, 0.57358333, 0.50076577, 0.45324175, 0.43439922,
         0.41857335, 0.38831392

In [ ]:
video_path = f'{path_to_mtmc_train}/train/S03/c010/vdo.avi'
video_player = VideoPlayer(video_path)
video_player.plot(start_frame=95, end_frame=120, bounding_boxes_gt=annotations)

In [ ]:
video_path = f'{path_to_mtmc_train}/train/S03/c010/vdo.avi'
video_player = VideoPlayer(video_path)
video_player.plot(start_frame=1, end_frame=400, bounding_boxes_gt=annotations, bounding_boxes_pred=tracking, save_file = "lastweek/mtsc_tc_mask_rcnn1.mp4")
video_player.plot(start_frame=401, end_frame=800, bounding_boxes_gt=annotations, bounding_boxes_pred=tracking, save_file = "lastweek/mtsc_tc_mask_rcnn2.mp4")
video_player.plot(start_frame=801, end_frame=1200, bounding_boxes_gt=annotations, bounding_boxes_pred=tracking, save_file = "lastweek/mtsc_tc_mask_rcnn3.mp4")
video_player.plot(start_frame=1201, end_frame=1600, bounding_boxes_gt=annotations, bounding_boxes_pred=tracking, save_file = "lastweek/mtsc_tc_mask_rcnn4.mp4")
video_player.plot(start_frame=1601, end_frame=2041, bounding_boxes_gt=annotations, bounding_boxes_pred=tracking, save_file = "lastweek/mtsc_tc_mask_rcnn5.mp4")

## S01

In [ ]:
def parse_numframe_file(filepath):
    result = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            key, value = line.split()
            result[key] = int(value)
    return result

In [ ]:
path_to_mtmc_train = "/Volumes/KINGSTON/MCV/datasets/C6/aic19-track1-mtmc-train"

### C001

In [105]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S01"
cam = "c001"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")
calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.93969217, 0.93969217, 0.93969217, 0.93952724, 0.93368807,
         0.90915812, 0.84165978, 0.74453587, 0.65641353, 0.52564865,
         0.37039087, 0.28562161, 0.22140301, 0.16268699, 0.10836076,
         0.06628224, 0.04823099, 0.01863772, 0.        ]),
  'DetA': array([0.9324192 , 0.9324192 , 0.9324192 , 0.93178851, 0.91988323,
         0.89286856, 0.82067056, 0.71019936, 0.60449986, 0.45215898,
         0.27592153, 0.18309015, 0.12378963, 0.07403375, 0.03569554,
         0.01578857, 0.0071465 , 0.00152284, 0.        ]),
  'AssA': array([0.94702188, 0.94702188, 0.94702188, 0.94733023, 0.94770007,
         0.92574486, 0.86318582, 0.78053247, 0.71278548, 0.61108264,
         0.49720438, 0.44557125, 0.39598866, 0.35749989, 0.32895018,
         0.2782605 , 0.32550586, 0.22810287, 0.        ]),
  'DetRe': array([0.9694501 , 0.9694501 , 0.9694501 , 0.96911066, 0.96266124,
         0.94772573, 0.90563476, 0.83435166, 0.75695859, 0.62559403,
         0.43448744, 0.31093007

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C002

In [106]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S01"
cam = "c002"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.78020336, 0.78020336, 0.78020336, 0.78020336, 0.76685732,
         0.74333506, 0.65140533, 0.53790658, 0.46389969, 0.35411454,
         0.29810909, 0.23046217, 0.17105445, 0.11144082, 0.06056571,
         0.02710587, 0.00964911, 0.00274025, 0.        ]),
  'DetA': array([8.06826707e-01, 8.06826707e-01, 8.06826707e-01, 8.06826707e-01,
         7.76835116e-01, 7.34293429e-01, 6.25168691e-01, 5.13589945e-01,
         4.33204403e-01, 2.91594047e-01, 2.14727021e-01, 1.48545541e-01,
         9.63923979e-02, 5.43942213e-02, 2.40221088e-02, 8.69018951e-03,
         2.70607827e-03, 6.23182385e-04, 0.00000000e+00]),
  'AssA': array([0.75445852, 0.75445852, 0.75445852, 0.75445852, 0.75700769,
         0.75248802, 0.67874305, 0.56337453, 0.49676994, 0.43004002,
         0.41386981, 0.35755239, 0.303547  , 0.22831572, 0.15270122,
         0.08454686, 0.03440602, 0.0120494 , 0.        ]),
  'DetRe': array([0.98715007, 0.98715007, 0.98715007, 0.98715007, 0.96649839,
         0.9359

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C003

In [107]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S01"
cam = "c003"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.87190115, 0.87190115, 0.87190115, 0.87190115, 0.86863668,
         0.84516453, 0.78327903, 0.6733987 , 0.58288896, 0.48536684,
         0.39297619, 0.31382274, 0.22523854, 0.16324657, 0.11124608,
         0.06381793, 0.03376895, 0.00242123, 0.        ]),
  'DetA': array([9.18725869e-01, 9.18725869e-01, 9.18725869e-01, 9.18725869e-01,
         9.14659988e-01, 8.82386364e-01, 8.01196086e-01, 6.17149365e-01,
         5.00452899e-01, 3.62253289e-01, 2.61134374e-01, 1.73851423e-01,
         1.07409471e-01, 6.16321299e-02, 3.12305458e-02, 1.35631246e-02,
         5.15776699e-03, 3.01932367e-04, 0.00000000e+00]),
  'AssA': array([0.82746294, 0.82746294, 0.82746294, 0.82746294, 0.82492915,
         0.80951284, 0.76576265, 0.73477481, 0.67890412, 0.65032114,
         0.59138245, 0.5664878 , 0.47232706, 0.43239532, 0.3962688 ,
         0.30027949, 0.22109217, 0.01941609, 0.        ]),
  'DetRe': array([9.71819481e-01, 9.71819481e-01, 9.71819481e-01, 9.71819481e-01,
         9.

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C004

In [108]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S01"
cam = "c004"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.95024882, 0.95024882, 0.95024882, 0.95024882, 0.94014643,
         0.91150098, 0.82634917, 0.7431165 , 0.68140916, 0.63879203,
         0.54091862, 0.46346785, 0.23541782, 0.15883051, 0.1063109 ,
         0.06345606, 0.03020577, 0.00351402, 0.        ]),
  'DetA': array([9.38948559e-01, 9.38948559e-01, 9.38948559e-01, 9.38948559e-01,
         9.27688273e-01, 8.93981226e-01, 8.02417236e-01, 7.01670250e-01,
         6.24822359e-01, 5.68119476e-01, 4.55445545e-01, 3.55017119e-01,
         1.46901471e-01, 6.87577898e-02, 2.77666800e-02, 1.32939439e-02,
         5.17729804e-03, 2.91630213e-04, 0.00000000e+00]),
  'AssA': array([0.96168509, 0.96168509, 0.96168509, 0.96168509, 0.9527719 ,
         0.92936407, 0.85099486, 0.78701089, 0.74312072, 0.71825607,
         0.64243236, 0.60504813, 0.37727023, 0.36689852, 0.4070349 ,
         0.30289519, 0.17622868, 0.04234234, 0.        ]),
  'DetRe': array([9.95405513e-01, 9.95405513e-01, 9.95405513e-01, 9.95405513e-01,
         9.

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C005

In [109]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S01"
cam = "c005"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.91542247, 0.91542247, 0.91542247, 0.91542247, 0.8828594 ,
         0.73490381, 0.60339587, 0.41320735, 0.32162125, 0.24722225,
         0.19091667, 0.14118694, 0.10938835, 0.0742362 , 0.03189997,
         0.01785459, 0.00588098, 0.00167256, 0.        ]),
  'DetA': array([8.80381471e-01, 8.80381471e-01, 8.80381471e-01, 8.80381471e-01,
         8.41740059e-01, 6.45838302e-01, 5.04797209e-01, 3.08246445e-01,
         2.12403373e-01, 1.52471610e-01, 1.06816359e-01, 7.09186840e-02,
         4.07178404e-02, 2.34317070e-02, 1.00995316e-02, 4.95121596e-03,
         1.30586187e-03, 2.89897087e-04, 0.00000000e+00]),
  'AssA': array([0.95185817, 0.95185817, 0.95185817, 0.95185817, 0.92598743,
         0.83625207, 0.72125315, 0.55390845, 0.48699899, 0.4008539 ,
         0.34123213, 0.281079  , 0.29387148, 0.23519471, 0.10075795,
         0.06438547, 0.02648516, 0.00964985, 0.        ]),
  'DetRe': array([9.06311360e-01, 9.06311360e-01, 9.06311360e-01, 9.06311360e-01,
         8.

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

## S03

In [ ]:
path_to_mtmc_train = "/Volumes/KINGSTON/MCV/datasets/C6/aic19-track1-mtmc-train"

### C010

In [110]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S03"
cam = "c010"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")
calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.99401532, 0.99401532, 0.99401532, 0.99401532, 0.99401532,
         0.99401532, 0.96698715, 0.88762211, 0.79645592, 0.70024755,
         0.59610681, 0.49364766, 0.31706899, 0.20147077, 0.13112511,
         0.07168231, 0.01967002, 0.00282426, 0.        ]),
  'DetA': array([9.93572576e-01, 9.93572576e-01, 9.93572576e-01, 9.93572576e-01,
         9.93572576e-01, 9.93572576e-01, 9.65153115e-01, 8.77901110e-01,
         7.76610979e-01, 6.75821702e-01, 5.56670849e-01, 4.34849653e-01,
         2.49832102e-01, 1.35794934e-01, 7.20046083e-02, 3.33148251e-02,
         8.12567714e-03, 8.06668459e-04, 0.00000000e+00]),
  'AssA': array([0.99445827, 0.99445827, 0.99445827, 0.99445827, 0.99445827,
         0.99445827, 0.96882468, 0.89745074, 0.81680797, 0.72555621,
         0.63833651, 0.56039601, 0.40240123, 0.29891004, 0.23878742,
         0.15423625, 0.04761568, 0.00988813, 0.        ]),
  'DetRe': array([0.99946121, 0.99946121, 0.99946121, 0.99946121, 0.99946121,
         0.9994

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C011

In [111]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S03"
cam = "c011"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.9151022 , 0.9151022 , 0.9151022 , 0.9151022 , 0.88903429,
         0.8403706 , 0.79504827, 0.75534895, 0.69900752, 0.59833278,
         0.52485154, 0.44495866, 0.36448329, 0.30076639, 0.25163278,
         0.18032961, 0.09882808, 0.02707158, 0.00536904]),
  'DetA': array([0.93955461, 0.93955461, 0.93955461, 0.93955461, 0.91317992,
         0.85685279, 0.80019685, 0.74689589, 0.6795225 , 0.57265692,
         0.48820179, 0.39511823, 0.30177936, 0.21366954, 0.15321564,
         0.08610451, 0.03743619, 0.0077135 , 0.00109469]),
  'AssA': array([0.89128618, 0.89128618, 0.89128618, 0.89128618, 0.86552711,
         0.82420546, 0.78993282, 0.76389768, 0.71905126, 0.62515986,
         0.56425261, 0.50108599, 0.44021589, 0.42336601, 0.41326758,
         0.37766625, 0.26089698, 0.09501145, 0.02633311]),
  'DetRe': array([0.98553949, 0.98553949, 0.98553949, 0.98553949, 0.97107898,
         0.93882091, 0.90433815, 0.86985539, 0.82313682, 0.74082314,
         0.66740823, 0.57619577

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C012

In [112]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S03"
cam = "c012"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.77859978, 0.77859978, 0.77859978, 0.77859978, 0.77859978,
         0.77859978, 0.77859978, 0.77859978, 0.77859978, 0.77859978,
         0.75989203, 0.72826787, 0.66296006, 0.59455559, 0.35780183,
         0.24971617, 0.18772106, 0.09858241, 0.        ]),
  'DetA': array([0.60621762, 0.60621762, 0.60621762, 0.60621762, 0.60621762,
         0.60621762, 0.60621762, 0.60621762, 0.60621762, 0.60621762,
         0.58974359, 0.55778894, 0.47619048, 0.3963964 , 0.1969112 ,
         0.09929078, 0.05442177, 0.02649007, 0.        ]),
  'AssA': array([1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 1.        , 1.        , 1.        ,
         0.97913043, 0.95085085, 0.92298368, 0.89177489, 0.6501517 ,
         0.62803583, 0.64752024, 0.36687307, 0.        ]),
  'DetRe': array([0.76470588, 0.76470588, 0.76470588, 0.76470588, 0.76470588,
         0.76470588, 0.76470588, 0.76470588, 0.76470588, 0.76470588,
         0.75163399, 0.7254902 

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C013

In [113]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S03"
cam = "c013"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.97698527, 0.97698527, 0.97698527, 0.97698527, 0.97009386,
         0.91172971, 0.85188229, 0.80794358, 0.71202151, 0.60157406,
         0.51256425, 0.42791983, 0.34986101, 0.25355113, 0.17362698,
         0.11609474, 0.07643698, 0.01938361, 0.00232346]),
  'DetA': array([9.98673740e-01, 9.98673740e-01, 9.98673740e-01, 9.98673740e-01,
         9.88126649e-01, 9.07594937e-01, 8.26666667e-01, 7.79220779e-01,
         6.80044593e-01, 5.73068894e-01, 4.78900883e-01, 3.80036630e-01,
         2.80373832e-01, 1.81960784e-01, 1.12177122e-01, 5.97749648e-02,
         2.79672578e-02, 6.00801068e-03, 6.64010624e-04]),
  'AssA': array([0.95576781, 0.95576781, 0.95576781, 0.95576781, 0.95239016,
         0.91588333, 0.87786707, 0.83772513, 0.74550203, 0.6314971 ,
         0.5485939 , 0.48183615, 0.43656973, 0.35330787, 0.26873863,
         0.22547881, 0.20890898, 0.06253725, 0.00813008]),
  'DetRe': array([1.        , 1.        , 1.        , 1.        , 0.99468792,
         0.9521

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C014

In [114]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S03"
cam = "c014"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([0.96250878, 0.96250878, 0.96250878, 0.96250878, 0.95870834,
         0.94958759, 0.84563244, 0.66148486, 0.51895837, 0.41784479,
         0.3521558 , 0.26507077, 0.19480047, 0.12185196, 0.05938639,
         0.02525526, 0.01069548, 0.        , 0.        ]),
  'DetA': array([0.97158427, 0.97158427, 0.97158427, 0.97158427, 0.95938104,
         0.94210123, 0.82325414, 0.6291412 , 0.49410029, 0.39724138,
         0.33641161, 0.24969159, 0.17571959, 0.1083151 , 0.05104794,
         0.02137528, 0.00715848, 0.        , 0.        ]),
  'AssA': array([0.95351807, 0.95351807, 0.95351807, 0.95351807, 0.9580361 ,
         0.95713344, 0.86861904, 0.69549128, 0.54506705, 0.43951682,
         0.36863682, 0.2813972 , 0.21595329, 0.13708061, 0.06908689,
         0.02983953, 0.01598011, 0.        , 0.        ]),
  'DetRe': array([1.        , 1.        , 1.        , 1.        , 0.99358974,
         0.984375  , 0.91626603, 0.78365385, 0.67107372, 0.57692308,
         0.51081731, 0.40544872

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )

### C015

In [115]:
trackname = "mtsc_tc_mask_rcnn"
seq = "S03"
cam = "c015"
tracking = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/mtsc/{trackname}.txt")#

annotations = parse_tracking_file(f"{path_to_mtmc_train}/train/{seq}/{cam}/gt/gt.txt")

calculate_metrics_taking_only_GTobject_into_account(tracking, annotations)
# calculate_metrics(tracking, annotations)


Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          


({'HOTA': array([1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 0.7       , 0.61904762, 0.36      ,
         0.25925926, 0.09677419, 0.        , 0.        ]),
  'DetA': array([1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 0.7       , 0.61904762, 0.36      ,
         0.25925926, 0.09677419, 0.        , 0.        ]),
  'AssA': array([1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 0.7       , 0.61904762, 0.36      ,
         0.25925926, 0.09677419, 0.        , 0.        ]),
  'DetRe': array([1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        

* vis only pred objects related to GT

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=True,
        save_file=save_file
    )

* vis all preds

In [ ]:
numframe = parse_numframe_file(f"{path_to_mtmc_train}/cam_framenum/{seq}.txt")[cam]
video_path = f'{path_to_mtmc_train}/train/{seq}/{cam}/vdo.avi'
video_player = VideoPlayer(video_path)
num_chunks = 5
chunk_size = numframe // num_chunks  # integer division

for i in range(num_chunks):
    # Compute start and end frame for chunk i (frames are assumed to start at 1)
    start_frame = i * chunk_size + 1
    # For the last chunk, ensure we go to the end of the video
    if i == num_chunks - 1:
        end_frame = numframe + 1  # +1 because end_frame is exclusive
    else:
        end_frame = (i + 1) * chunk_size + 1

    save_file = f"{seqcam}/{trackname}{i+1}_allpred.mp4"
    print(f"Chunk {i+1}: start_frame={start_frame}, end_frame={end_frame}, saving to {save_file}")

    video_player.plot(
        start_frame=start_frame,
        end_frame=end_frame,
        bounding_boxes_gt=annotations,
        bounding_boxes_pred=tracking,
        taking_only_GTobject_into_account=False,
        save_file=save_file
    )